<a href="https://colab.research.google.com/github/AngryWeassel/Proyecto_Tasa_de_Natalidad.ipynb/blob/main/Proyecto_Tasa_de_Natalidad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import gdown
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
import unicodedata

#Importación de lista de Paises de la OCDE (En ingles OECD)
#Fuente: Consulta en Google y creación de Excel manual
archivo_nombres_ocde = 'https://docs.google.com/spreadsheets/d/1gpatuWv1C5Iq14sTngbqZV01PtK4T-1VuIty43tDE58/edit?usp=drive_link'
nombres_ocde = gdown.download(archivo_nombres_ocde, quiet=True, fuzzy=True)
ocde = pd.read_excel(nombres_ocde)

#Importación de datos del PIB de los paises a nivel global
#Fuente: https://ourworldindata.org/grapher/gdp-per-capita-maddison-project-database
archivo_pib = 'https://drive.google.com/file/d/1RWhkgYFs73Ya0XUdyNmeo8IZ7yLC7mxG/view?usp=sharing'
pib = gdown.download(archivo_pib, quiet=True, fuzzy=True)
datos_pib = pd.read_csv(pib)

#Importación de datos de Fertilidad de los paises pertenecientes a la OCDE (filtrado manualmente desde la fuente)
#Fuente: https://ourworldindata.org/global-decline-fertility-rate
archivo_fertilidad = 'https://drive.google.com/file/d/1dmf8CR_6edDLQi1DRM9Rr_PhI7JW4afP/view?usp=sharing'
fertilidad = gdown.download(archivo_fertilidad, quiet=True, fuzzy=True)
datos_fertilidad = pd.read_csv(fertilidad)

#Importación de datos de las edades de retiro (pensión) de los paises de la OCDE por cada hoja (hoja "Hombres" y hoja "Mujeres")
#Fuente: https://docs.google.com/document/d/1_BfpFvd0QlCehMby4jK3u5rP25SyxnaNoGKUo5DasRo/edit?usp=sharing
archivo_pensiones = 'https://docs.google.com/spreadsheets/d/1xEL0RQ5GfFZG0gBU6bYaIjlXI3QUtN6hni_eN4HLuBk/edit?usp=sharing'
pensiones_descarga = gdown.download(archivo_pensiones, quiet=True, fuzzy=True)
pensiones_hombres = pd.read_excel(pensiones_descarga, sheet_name='Hombres')
pensiones_mujeres = pd.read_excel(pensiones_descarga, sheet_name='Mujeres')

In [2]:
diccionario_paises = {
    'Alemania': 'Germany',
    'Australia': 'Australia',
    'Austria': 'Austria',
    'Bélgica': 'Belgium',
    'Canadá': 'Canada',
    'Chile': 'Chile',
    'Colombia': 'Colombia',
    'Corea del Sur': 'South Korea',
    'Costa Rica': 'Costa Rica',
    'Dinamarca': 'Denmark',
    'Eslovenia': 'Slovenia',
    'España': 'Spain',
    'Estados Unidos': 'United States',
    'Estonia': 'Estonia',
    'Finlandia': 'Finland',
    'Francia': 'France',
    'Grecia': 'Greece',
    'Hungría': 'Hungary',
    'Irlanda': 'Ireland',
    'Islandia': 'Iceland',
    'Israel': 'Israel',
    'Italia': 'Italy',
    'Japón': 'Japan',
    'Letonia': 'Latvia',
    'Lituania': 'Lithuania',
    'Luxemburgo': 'Luxembourg',
    'México': 'Mexico',
    'Noruega': 'Norway',
    'Nueva Zelanda': 'New Zealand',
    'Países Bajos': 'Netherlands',
    'Polonia': 'Poland',
    'Portugal': 'Portugal',
    'Reino Unido': 'United Kingdom',
    'República Checa': 'Czechia',
    'República Eslovaca': 'Slovakia',
    'Suecia': 'Sweden',
    'Suiza': 'Switzerland',
    'Turquía': 'Turkey'
}
ocde['OCDE paises (ingles)'] = ocde['Paises OCDE'].map(diccionario_paises)
inv_diccionario_paises = {v: k for k, v in diccionario_paises.items()}

diccionario_continentes = {
    'north america': 'américa del norte',
    'south america': 'américa del sur',
    'europe': 'europa',
    'asia': 'asia',
    'africa': 'áfrica',
    'oceania': 'oceanía'
}

def eliminar_tildes(texto):
    if not isinstance(texto, str):
        return texto  # Si no es string, lo devuelve igual
    # Normaliza a forma NFD (descompone letras y acentos)
    texto_normalizado = unicodedata.normalize('NFD', texto)
    # Filtra caracteres que no sean marcas diacríticas
    texto_sin_tildes = ''.join(
        c for c in texto_normalizado if unicodedata.category(c) != 'Mn'
    )
    return texto_sin_tildes


In [3]:
datos_pib = datos_pib.rename(columns={'Entity' : 'pais', 'Year' : 'año','GDP per capita': 'pib por pais', 'World region according to OWID' : 'continente'})
datos_pib = datos_pib.drop('Code', axis=1)

datos_fertilidad = datos_fertilidad.rename(columns={'Entity' : 'pais', 'Year' : 'año','Total fertility rate': 'fertilidad por pais', 'World region according to OWID' : 'continente'})
datos_fertilidad = datos_fertilidad.drop('Code', axis=1)

pensiones_hombres = pensiones_hombres.rename(columns={'Country' : 'pais'})
pensiones_mujeres = pensiones_mujeres.rename(columns={'Country' : 'pais'})

In [12]:
lista_ocde = ocde['OCDE paises (ingles)'].tolist()

#Filtración de datos con los parametros de la lista de paises de la OCDE y los años de 2008 a 2024, adicionalmente se crea una copia
ocde_pib = datos_pib[
    (datos_pib['pais'].isin(lista_ocde)) &
    (datos_pib['año'] >= 2008) &
    (datos_pib['año'] <= 2024)
].copy()

# Proceso para ocde_pib
# 1. Se crea una nueva columna usando la función LAMBDA para "traducir" los nombres de los paises
# 2. Se elimina la columna llamada "pais" la cual contienen los paises en ingles
# 3. La columna con los paises en español es renombra a "pais"
# 4. Se reorganiza el orden de la columna con los paises para ubicarse en el primer lugar del marco de datos
# 5. Se cambian todos los datos de la primer columna (paises) a minusculas y se eliminan las tildes
# 6. Se cambian todos los datos de la ultima columna (continentes) usando la función LAMBDA se "traducen" los nombres de los continentes a español, se cabia a minusculas y se eliminan las tildes
ocde_pib['pais español'] = ocde_pib['pais'].apply(lambda x: inv_diccionario_paises.get(x, x))
ocde_pib = ocde_pib.drop(columns=['pais'])
ocde_pib = ocde_pib.rename(columns={'pais español': 'pais'})
col_ocde_pib = ocde_pib.columns.tolist()
col_ocde_pib = ['pais'] + [col for col in col_ocde_pib if col != 'pais']
ocde_pib = ocde_pib[col_ocde_pib]
ocde_pib['pais'] = ocde_pib['pais'].str.lower().apply(eliminar_tildes)
ocde_pib['continente'] = ocde_pib['continente'].str.lower().apply(lambda x: diccionario_continentes.get(eliminar_tildes(x), x))

# Proceso para pensiones_hombres
# 1. Se crea una nueva columna usando la función LAMBDA para "traducir" los nombres de los paises
# 2. Se elimina la columna llamada "pais" la cual contienen los paises en ingles
# 3. La columna con los paises en español es renombra a "pais"
# 4. Se reorganiza el orden de la columna con los paises para ubicarse en el primer lugar del marco de datos
# 5. Se cambian todos los datos de la primer columna (paises) a minusculas y se eliminan las tildes
pensiones_hombres['pais español'] = pensiones_hombres['pais'].apply(lambda x: inv_diccionario_paises.get(x, x))
pensiones_hombres = pensiones_hombres.drop(columns=['pais'])
pensiones_hombres = pensiones_hombres.rename(columns={'pais español': 'pais'})
col_hombres = pensiones_hombres.columns.tolist()
col_hombres = ['pais'] + [col for col in col_hombres if col != 'pais']
pensiones_hombres = pensiones_hombres[col_hombres]
pensiones_hombres['pais'] = pensiones_hombres['pais'].str.lower().apply(eliminar_tildes)

# Proceso para pensiones_mujeres
# 1. Se crea una nueva columna usando la función LAMBDA para "traducir" los nombres de los paises
# 2. Se elimina la columna llamada "pais" la cual contienen los paises en ingles
# 3. La columna con los paises en español es renombra a "pais"
# 4. Se reorganiza el orden de la columna con los paises para ubicarse en el primer lugar del marco de datos
# 5. Se cambian todos los datos de la primer columna (paises) a minusculas y se eliminan las tildes
pensiones_mujeres['pais español'] = pensiones_mujeres['pais'].apply(lambda x: inv_diccionario_paises.get(x, x))
pensiones_mujeres = pensiones_mujeres.drop(columns=['pais'])
pensiones_mujeres = pensiones_mujeres.rename(columns={'pais español': 'pais'})
col_mujeres = pensiones_mujeres.columns.tolist()
col_mujeres = ['pais'] + [col for col in col_mujeres if col != 'pais']
pensiones_mujeres = pensiones_mujeres[col_mujeres]
pensiones_mujeres['pais'] = pensiones_mujeres['pais'].str.lower().apply(eliminar_tildes)

In [13]:
ocde_pib

,pais,año,pib por pais,continente
322,australia,2008,52473.383,oceania
323,australia,2009,52407.910,oceania
324,australia,2010,52739.270,oceania
325,australia,2011,53257.047,oceania
326,australia,2012,54403.883,oceania
...,...,...,...,...
6934,estados unidos,2020,67342.070,america del norte
6935,estados unidos,2021,71307.400,america del norte
6936,estados unidos,2022,72679.260,america del norte
6937,estados unidos,2023,74158.720,america del norte


In [ ]:
pais_pib_sum = ocde_pib.groupby('pais')['pib por pais'].mean()
plt.figure(figsize=(20, 10))
pais_pib_sum.plot(kind='bar', color='skyblue')
plt.title('Promedio total PIB por pais (2008-2024) [OECD]')
plt.xlabel('Pais')
plt.ylabel('Total PIB por pais (suma en miles de dolares $)')
plt.xticks(rotation=75, fontsize=9)
plt.grid(axis='y', linestyle='--', alpha=0.7)

total_pib_promedio = ocde_pib['pib por pais'].mean()

plt.axhline(y=total_pib_promedio, color='gray', linestyle='--', label=f'Total Promedio PIB [OCDE]: $ {total_pib_promedio:,.0f}')

for index, value in enumerate(pais_pib_sum):
    plt.text(index, value + (max(pais_pib_sum) * 0.0005), f'{value:,.0f}', ha='center', va='bottom', fontsize=8)

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Save to Excel
file_name3 = 'data_pib_original.xlsx'
ocde_pib.to_excel(file_name3, index=False)

# Download to local machine
#files.download(file_name3)

# Alternative to: ocde_pib['pais español'] = ocde_pib['pais'].map(inv_diccionario_paises)
# This alternative code snippet would typically be used in cell yauG_3_kQd_E
# ocde_pib['pais español'] = ocde_pib['pais'].map(inv_diccionario_paises)
# ocde_pib['pais español'] = ocde_pib['pais'].apply(lambda x: inv_diccionario_paises.get(x, x))